In [7]:
import numpy as np
import sys, os, time
import torch
import torch.nn as nn
import data, architecture
import optuna
from torch.utils.data import random_split, DataLoader

## Load BSQ model and pass beyond lcdm Pks through it

In [8]:
################################### INPUT ############################################
# data parameters
f_Pk_norm = None #file with Pk to normalize Pk
seed      = 42         #seed to split data in train/valid/test
mode      = 'all'   #'train','valid','test' or 'all'
additional_extension = '_fixed'
final_hidden_layer_size = 10   
log = True

all_n_sims_BSQ = [2000, 4000, 8000, 16000, 22000]

In [11]:
for n_sims_BSQ in all_n_sims_BSQ:

    cosm_type = 'BSQ'
    Pk_type = 'Pk'
    params_ext = ''  #_lcdm or _MGfor fR
    
    log = True
    output_size = 5
    input_size  = 79  #dimensions of input data   
    
    name = 'transfer10_network1_'+str(n_sims_BSQ)+'_BSQ' + additional_extension
    f_Pk       = 'Pk_files/'+'all_'+str(Pk_type)+'_'+str(cosm_type)+params_ext+'.npy'
    f_params  = '../real_params/'+'all_'+str(cosm_type)+'_params'+params_ext+'.txt' 
    
    # f_Pk      = 'Pk_files/'+'all_'+cosm_type+'_'+str(n_sims_BSQ)+'_BSQ_'+Pk_type+params_ext+'_'+str(extension)+'_fortransfer_fixed.npy'
    study_name = str(Pk_type)+'_'+str(cosm_type)+'_params_'+str(name)
    mother = '/scratch/network/vk9342/USRP2024_scratch/pytorch/'+str(Pk_type)+'_'+str(cosm_type)+'/'+str(name)+'/models/'

    print('\n'+name)
    print(f_params)
    print(f_Pk)
    print(study_name)

    # architecture parameters
    # training parameters
    batch_size = 32
    # optuna parameters
    storage    = 'sqlite:///nwLH.db'

######################################################################################

    # use GPUs if available
    if torch.cuda.is_available():
        print("CUDA Available")
        device = torch.device('cuda')
    else:
        print('CUDA Not Available')
        device = torch.device('cpu')
    
    # load the optuna study
    study = optuna.load_study(study_name=study_name, storage=storage)
    
    # get the scores of the study trials
    values = np.zeros(len(study.trials))
    completed = 0
    for i,t in enumerate(study.trials):
        values[i] = t.value
        if t.value is not None:  completed += 1
    
    # get the info of the best trial
    indexes = np.argsort(values)
    for i in [0]:  #choose the best-model here, e.g. [0], or [1]
        trial = study.trials[indexes[i]]
        print("\nTrial number {}".format(trial.number))
        print("Value: %.5e"%trial.value)
        print(" Params: ")
        for key, value in trial.params.items():
            print("    {}: {}".format(key, value))
        lr       = trial.params['lr']
        wd       = trial.params['wd']
        n_layers       = trial.params['n_layers']
        p       = trial.params['dropout_l']
        
        '''comment h1 and dr out for dynamic models'''
        # h1       = trial.params['h1']
        # dr       = trial.params['dr']
        fmodel = mother +'model_%d.pt'%trial.number
    
    # # generate the architecture
    out_fs = [trial.params[f'n_units_l{i}'] for i in range(n_layers-1)]
    model = architecture.dynamic_model_fixed_final(trial, input_size, output_size, 
                                                   final_hidden_layer_size, 
                                                   n_layers, p, out_fs,
                                                   max_neurons_layers=500)
    model.to(device)  
    
    # load best-model, if it exists
    if os.path.exists(fmodel):  
        print('Loading model...')
        model.load_state_dict(torch.load(fmodel, map_location=torch.device(device)))
    else:
        raise Exception('model doesnt exists!!!')
    
    
    # define loss function
    criterion = nn.MSELoss() 
    test_loader = data.create_dataset(mode, seed, f_Pk, f_Pk_norm, f_params, 
                                  batch_size, shuffle=False, workers=1, 
                                  cosm_type =cosm_type, log=log, 
                                  shuffle_all = True)
    
        
    
    test_points = 0
    for x,y in test_loader:  test_points += x.shape[0]
    
    # define the arrays containing the true and predicted value of the parameters
    params  = output_size
    results = np.zeros((test_points, 2*params), dtype=np.float32)
    
    # test the model
    test_loss, points = 0.0, 0
    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            bs   = x.shape[0]  #batch size
            x, y = x.to(device), y.to(device)
            y_NN = model(x)
            test_loss += (criterion(y_NN, y).item())*bs
            results[points:points+bs,0*params:1*params] = y.cpu().numpy()
            results[points:points+bs,1*params:2*params] = y_NN.cpu().numpy()
            points    += bs
    test_loss /= points
    print('Test loss:', test_loss)

######################################################################################
######################################################################################
    ## Pass beyond LCDM through network 
    Pk_type = 'Pk'
    cosm_type = 'fR'
    
    if cosm_type == 'fR': 
        params_ext = '_lcdm'
    else:
        params_ext = ''

    f_Pk      = 'Pk_files/'+'all_'+Pk_type+'_'+cosm_type + params_ext + '.npy'        #file with Pk        #file with Pk
    f_params  = '../real_params/' +'all_' + cosm_type+'_params'+params_ext+'.txt'       #file with parameters
    
    extension = 'fhl10'
    fout = 'Pk_files/'+'all_'+cosm_type+'_'+str(n_sims_BSQ)+'_BSQ_'+Pk_type + params_ext+'_'+str(extension)+'_fortransfer'+ additional_extension+'.npy'
    
    test_loader = data.create_dataset(mode, seed, f_Pk, None, f_params, 
                                      batch_size, shuffle=False, 
                                      workers=1, cosm_type='BSQ', log=log, 
                                      shuffle_all= False)
    # Store results
    all_outputs = []
    true_values = []
    
    for x, y in test_loader:
        x = x.to(device)  # Move to device
        y = y.to(device)  # True cosmological parameters
        outputs = model[:-2](x)  # Pass through the final hidden layer
        # outputs = model(x)
        all_outputs.append(outputs.detach().cpu())  # Collect predictions
        true_values.append(y.detach().cpu())  # Collect true values
    
    # Combine batches
    all_outputs = torch.cat(all_outputs, dim=0)  # Predicted values: shape [N, 5]
    true_values = torch.cat(true_values, dim=0)  # True values: shape [N, 5]
    
    
    # Convert to NumPy
    all_outputs_np = all_outputs.numpy()
    true_values_np = true_values.numpy()
    print(all_outputs.shape)

    np.save(fout, all_outputs_np)
    print(fout)

 

    




transfer10_network1_2000_BSQ_fixed
../real_params/all_BSQ_params.txt
Pk_files/all_Pk_BSQ.npy
Pk_BSQ_params_transfer10_network1_2000_BSQ_fixed
CUDA Available

Trial number 40
Value: 2.04760e-02
 Params: 
    lr: 0.0023207986079189355
    wd: 1.2304600572865334e-07
    n_layers: 2
    dropout_l: 0.23170167880162712
    n_units_l0: 370
Loading model...
Test loss: 0.02034615749772638
torch.Size([2000, 10])
Pk_files/all_fR_2000_BSQ_Pk_lcdm_fhl10_fortransfer_fixed.npy

transfer10_network1_4000_BSQ_fixed
../real_params/all_BSQ_params.txt
Pk_files/all_Pk_BSQ.npy
Pk_BSQ_params_transfer10_network1_4000_BSQ_fixed
CUDA Available

Trial number 41
Value: 1.59859e-02
 Params: 
    lr: 0.000818865461204283
    wd: 4.4698855957202115e-06
    n_layers: 3
    dropout_l: 0.24684427119223543
    n_units_l0: 263
    n_units_l1: 432
Loading model...
Test loss: 0.015659166326746344
torch.Size([2000, 10])
Pk_files/all_fR_4000_BSQ_Pk_lcdm_fhl10_fortransfer_fixed.npy

transfer10_network1_8000_BSQ_fixed
../real_